# Anotación humana: calibración y tanda 1 del Test Gold

Este cuaderno prepara el trabajo de anotación del equipo a partir del jurado (`jurado_18000_v3.csv`):

1. **Calibración:** 20 citas de 2014 (partición val) que anotan las 5 personas, para alinear criterios antes del test.
2. **Tandas del Test Gold:** citas de 2015 (partición test), al azar por clase, cada una asignada a una pareja de anotadores. La tanda 1 es el mínimo que se necesita en cualquier escenario (tope de 500 por clase); las tandas 2 y 3 (topes de 1.000 y 2.000) quedan escogidas desde hoy para cuando el equipo avance.
3. **Traducción de apoyo:** contexto, título y resumen en español de la calibración y de las tres tandas, hechos por GPT en una sola corrida. Son solo una ayuda: **la etiqueta se decide con el texto original en inglés.**
4. **Hojas por anotador** en Excel, con el mismo formato de las calibraciones anteriores.

**Reglas de la anotación a ciegas** (guía `docs/guia_anotacion_funcion_cita.md`): las hojas no muestran la etiqueta de ningún modelo. Cada persona anota sola, sin ver las respuestas de su pareja. El archivo `clave_coordinador_v2.csv` trae las etiquetas del jurado y **no se comparte con los anotadores**.

**Exclusiones:** no entran citas ya vistas por el equipo (taller y calibraciones anteriores) ni las 200 citas usadas para escoger a los jueces, para no evaluar al jurado con citas con las que se ajustó.

In [ ]:
!pip install -q openai openpyxl

## Paso 1. Configuración y datos

In [ ]:
import csv, json, math, os, random, re, time                          # utilidades estándar
from collections import Counter, defaultdict                          # conteos por clase
from itertools import combinations                                    # parejas de anotadores
from pathlib import Path

CARPETA = Path(os.environ.get('CARPETA_JUECES', '/content/drive/MyDrive/proyecto_grado/jueces'))  # donde están el jurado y las citas
if str(CARPETA).startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')                                     # monta Drive en Colab
JURADO = CARPETA / 'jurado_18000_v3.csv'                              # voto del jurado por cita
CITAS = CARPETA / 'ANOTACION_18000_CITAS_REALES_QLORA.csv'            # textos: contexto, título y resumen
SALIDA = CARPETA / 'anotacion'                                        # aquí quedan las hojas y la clave
SALIDA.mkdir(parents=True, exist_ok=True)

SEMILLA = 2026            # fija el azar: con la misma semilla salen siempre las mismas citas
TOPES = {1: 500, 2: 1000, 3: 2000}   # tope por clase al terminar cada tanda: la 1 es el mínimo, la 3 la meta del enunciado
TANDA_ACTUAL = 1          # la tanda cuyas hojas de Excel se generan ahora (las traducciones se hacen para todas)
VERSION_HOJAS = 'v2'      # va en el nombre de las hojas y de la clave. v1: repartidas el 3-nov a las 7:43 p. m.; v2: con las exclusiones posteriores de John
FRACCION_TEST = 0.15      # el enunciado pide validar al menos el 15 % de cada clase
N_CALIBRACION = 20        # citas de la calibración
RESUMEN_MAX = 1200        # caracteres del resumen que se muestran y traducen (hay resúmenes mal extraídos muy largos)
PERSONAS = ['Arias', 'Caicedo', 'Duque', 'Pinzon', 'Zambrana']        # los 5 anotadores (sin tilde para los nombres de archivo)
CLASES = ['Background', 'Gap', 'Basis', 'Comparison', 'Application', 'Improvement / Modification', 'Evidence',
          'Identification of the Originator', 'Further Reading']      # las 9 funciones del enunciado
EVALUABLE = ['Evaluable', 'Ambiguo', 'No evaluable']                  # mismas opciones que el CSV del QLoRA

In [ ]:
EXCLUIR_TALLER = [  # citas usadas en el taller de práctica (proyecto_de_grado/anotacion/ids_excluir_test_gold.txt)
    'N10-1073_D08-1037_0', 'P07-1072_P06-2064_0', 'J01-3003_W99-0632_3', 'P09-1070_P06-1101_0',
    'W02-1027_P88-1012_1', 'D15-1187_D10-1036_0', 'P15-1127_P08-1039_0', 'Q13-1005_P09-1010_0',
    'D14-1066_N13-1133_2', 'N07-1051_P05-1022_5', 'W11-2912_P09-2056_0', 'P13-1135_W02-0109_0',
    'P11-1110_N04-1019_1', 'W10-1724_P07-2045_0', 'E06-1012_P96-1025_2', 'Q13-1005_D11-1039_1',
    'P15-1128_D14-1002_0', 'N04-1026_P01-1048_1', 'P10-1049_P08-2007_0', 'D13-1174_P12-2063_0',
    'W11-2128_N06-1003_1', 'W15-2104_N06-1058_0', 'W01-0712_W95-0107_0', 'S13-2014_P12-1010_1',
    'P12-2056_W08-0336_1', 'P10-1142_D09-1101_1', 'W14-4407_W13-2117_1', 'P10-1076_W04-3250_0',
    'P13-3025_W11-2103_0', 'P06-1122_N03-1017_0', 'P14-2008_N13-1067_2', 'P10-1110_P04-1015_0',
    'W15-3012_N03-1017_0', 'P15-1029_W13-0109_0', 'S15-2047_S15-2044_0', 'D15-1294_P14-2085_4',
    'S15-1021_N13-1090_0', 'W15-3037_D14-1162_0', 'W15-0807_P08-1090_0', 'J15-3004_P02-1026_0',
    'P15-1148_P10-1110_2', 'K15-1029_P96-1025_0', 'W05-0638_W04-3212_0', 'P13-1137_W12-1642_0',
    'P14-1066_D13-1176_0', 'J13-3003_W09-2413_0', 'W15-2212_S14-2082_2', 'E06-1040_E06-1045_0',
    'N01-1003_W98-1415_1', 'D14-1101_E14-4030_0', 'D13-1129_P08-1068_1', 'D15-1104_N03-1028_0',
    'P13-1009_N12-1049_5', 'S07-1072_P97-1023_0', 'W14-1005_W11-2126_1', 'E14-1037_D11-1140_1',
    'E12-1050_P07-1096_1', 'W11-0507_N09-1041_0', 'Q15-1006_N07-1055_0', 'E09-1061_J07-2003_3',
    'W13-3602_D10-1094_2', 'W05-0620_W04-3212_1', 'W11-2102_D10-1092_2', 'W09-1401_W04-1213_0',
    'P14-1135_P09-1110_0', 'W12-1614_P02-1047_2', 'N10-1120_W96-0213_0', 'W15-2117_P05-1013_0',
    'W13-3720_E06-1010_1', 'D14-1209_D13-1201_0', 'D13-1057_P12-1041_0', 'W11-2402_E09-1064_1',
    'P15-1043_N09-1066_0', 'W06-3121_P05-1066_0', 'W15-4641_W14-4337_1', 'P14-3013_P13-2054_1',
    'S15-2025_S13-1005_2',
]

EXCLUIR_CALIBRACIONES = [  # calibraciones anteriores (proyecto_de_grado/anotacion/taller/calib_ids.json)
    'K15-1029_P96-1025_0', 'W05-0638_W04-3212_0', 'P13-1137_W12-1642_0', 'P14-1066_D13-1176_0',
    'J13-3003_W09-2413_0', 'W15-2212_S14-2082_2', 'E06-1040_E06-1045_0', 'N01-1003_W98-1415_1',
    'D14-1101_E14-4030_0', 'D13-1129_P08-1068_1', 'D15-1104_N03-1028_0', 'P13-1009_N12-1049_5',
    'S07-1072_P97-1023_0', 'W14-1005_W11-2126_1', 'E14-1037_D11-1140_1', 'E12-1050_P07-1096_1',
    'W11-0507_N09-1041_0', 'Q15-1006_N07-1055_0', 'E09-1061_J07-2003_3', 'W13-3602_D10-1094_2',
    'W05-0620_W04-3212_1', 'W11-2102_D10-1092_2', 'W09-1401_W04-1213_0', 'P14-1135_P09-1110_0',
    'W12-1614_P02-1047_2', 'N10-1120_W96-0213_0', 'W15-2117_P05-1013_0', 'W13-3720_E06-1010_1',
    'D14-1209_D13-1201_0', 'D13-1057_P12-1041_0', 'W11-2402_E09-1064_1', 'P15-1043_N09-1066_0',
    'W06-3121_P05-1066_0', 'W15-4641_W14-4337_1', 'P14-3013_P13-2054_1', 'S15-2025_S13-1005_2',
]

EXCLUIR_PRUEBA_JUECES = [  # las 200 citas con las que se escogió el jurado (Paso 4 del cuaderno de jueces)
    'D09-1002_P06-4020_0', 'D09-1044_W04-1017_0', 'D09-1073_N04-4026_0', 'D09-1073_P07-1089_0',
    'D09-1086_D08-1017_0', 'D09-1111_W06-1672_0', 'D09-1115_N03-1017_0', 'D10-1042_P09-1098_1',
    'D10-1108_P08-1078_0', 'D11-1001_P05-1073_0', 'D11-1045_N04-4026_0', 'D11-1091_D10-1114_1',
    'D11-1097_P10-1044_0', 'D12-1003_W11-1206_1', 'D12-1015_P10-1139_0', 'D12-1038_P11-2126_0',
    'D12-1047_D08-1090_0', 'D12-1056_P02-1019_0', 'D13-1064_P12-1075_0', 'D13-1134_D10-1040_0',
    'D13-1155_P10-1058_0', 'D13-1173_P11-1002_1', 'D14-1037_P13-1155_3', 'D14-1087_N04-1041_2',
    'D14-1109_D10-1125_1', 'D14-1127_N12-1071_0', 'D14-1218_P11-1100_1', 'D15-1143_N10-1028_1',
    'D15-1148_P11-2111_5', 'D15-1186_D14-1003_0', 'E03-1017_P02-1013_0', 'E03-1076_P01-1030_0',
    'E09-1017_P03-1069_0', 'E09-1018_P07-1107_0', 'E12-1003_D11-1122_4', 'E14-1070_W13-3509_1',
    'E95-1020_P90-1031_0', 'E97-1014_P96-1036_1', 'E97-1063_P97-1039_2', 'J05-4002_P98-2127_0',
    'J08-1002_W03-0401_0', 'J08-3003_P03-1054_0', 'J08-4004_W04-0210_0', 'J12-1003_W04-3205_0',
    'J13-4004_D09-1120_1', 'J14-2008_E12-1017_6', 'J14-4005_Q14-1019_0', 'J15-3005_N10-1115_0',
    'J96-2002_E91-1023_1', 'J97-3002_J90-2002_0', 'K15-2007_D09-1036_0', 'N04-4025_W01-0514_0',
    'N06-3005_J04-3002_0', 'N07-1037_P05-1017_0', 'N07-2045_N04-2006_0', 'N09-1070_N04-1019_0',
    'N10-1005_N04-1011_4', 'N10-1019_W00-0708_0', 'N10-1118_E09-1055_1', 'N10-1129_P07-2045_2',
    'N10-1145_W07-1401_3', 'N13-1037_D12-1039_0', 'N13-1053_P11-1032_6', 'N13-1055_W09-2112_0',
    'N13-1075_P08-1088_1', 'N13-1093_W09-1401_0', 'P00-1051_W99-0106_0', 'P01-1064_A00-2004_4',
    'P02-1036_J97-4005_0', 'P04-1064_W03-0301_1', 'P04-1066_W03-0301_0', 'P05-3027_W97-0322_0',
    'P07-1069_W03-0501_0', 'P07-1087_P03-1020_2', 'P07-1096_P04-1015_0', 'P08-1046_J99-3006_0',
    'P10-1001_E06-1011_6', 'P10-1049_W06-3105_1', 'P10-1078_P09-4001_0', 'P10-1112_P96-1024_0',
    'P11-1012_W06-1615_0', 'P11-1114_P98-1067_0', 'P11-1125_P03-1054_0', 'P11-1126_N10-1141_4',
    'P11-1126_P09-1064_0', 'P11-1128_D09-1076_3', 'P12-1023_P08-1076_1', 'P12-1028_N07-1071_0',
    'P12-1065_J04-3004_2', 'P12-1071_P09-1006_3', 'P13-1020_N04-1019_0', 'P13-1043_P11-2033_0',
    'P13-1092_P11-1149_0', 'P13-1103_P07-1031_0', 'P13-2012_P98-1013_0', 'P13-2056_N07-1057_0',
    'P13-2104_D09-1060_0', 'P13-2125_S12-1051_0', 'P14-1010_N12-1047_0', 'P14-1025_E12-1060_0',
    'P14-1034_N09-1068_0', 'P14-1049_S12-1035_2', 'P14-1092_P12-1007_0', 'P14-1092_P12-1007_2',
    'P14-1093_D11-1027_2', 'P14-1109_N09-1031_0', 'P14-2006_P14-2005_1', 'P14-2083_P14-2062_0',
    'P14-3003_W04-2703_0', 'P14-5002_P06-1005_0', 'P15-1010_D14-1162_0', 'P15-1019_E14-1024_0',
    'P15-1040_P13-1045_1', 'P15-1070_P10-1154_0', 'P15-1138_D13-1203_1', 'P15-2010_D11-1033_0',
    'P15-2053_P11-1081_0', 'P85-1023_J81-4003_0', 'P96-1011_E89-1003_0', 'P98-1112_A97-1028_0',
    'P99-1013_P99-1013_1', 'P99-1023_W96-0101_0', 'Q14-1010_Q13-1033_3', 'Q14-1033_P11-1092_0',
    'S12-1056_N13-4004_0', 'S12-1100_W11-2112_1', 'S15-2005_P14-5008_0', 'S15-2013_Q14-1034_2',
    'S15-2017_P07-2045_0', 'S15-2049_P04-1036_1', 'S15-2125_P08-1031_0', 'W00-1801_P96-1015_0',
    'W02-1024_A97-1030_0', 'W02-2010_N01-1006_0', 'W03-0302_W02-1039_1', 'W03-0508_W02-0406_0',
    'W03-2202_P98-2127_0', 'W03-2605_W97-1301_2', 'W04-0907_E03-1040_0', 'W04-3212_N04-1032_0',
    'W05-0803_W02-1039_0', 'W05-0906_P98-1013_0', 'W05-1510_P96-1027_0', 'W06-0103_W04-1102_3',
    'W06-0803_P04-1036_1', 'W06-1322_P03-1070_0', 'W06-2917_P85-1009_1', 'W07-0704_W06-3110_0',
    'W07-2202_W06-1619_0', 'W07-2305_E06-1045_0', 'W08-1122_W01-0521_1', 'W08-2004_P06-1051_2',
    'W09-0437_J07-2003_1', 'W09-0621_N06-1003_0', 'W09-1117_W06-2920_0', 'W09-3520_N07-1046_0',
    'W10-0403_W03-2502_0', 'W10-1404_P08-1108_1', 'W10-2101_W07-1317_0', 'W10-2406_W09-3510_0',
    'W10-3018_P04-1014_0', 'W10-3101_P07-1125_0', 'W10-4102_P09-1095_0', 'W10-4143_N07-1051_0',
    'W10-4205_P87-1002_0', 'W11-1716_P97-1023_0', 'W11-1902_P02-1014_0', 'W11-3708_D08-1014_2',
    'W12-1511_W11-2821_0', 'W12-1637_W11-2006_2', 'W12-3148_W04-3250_0', 'W12-3709_N03-1017_0',
    'W12-5105_W04-3221_0', 'W12-5503_P09-3003_1', 'W12-5903_W99-0604_0', 'W12-5906_D11-1045_0',
    'W12-6315_W03-1709_0', 'W13-1201_P11-2037_0', 'W13-2018_W11-1802_0', 'W13-2904_J10-2002_0',
    'W13-3513_P08-1028_2', 'W13-5305_A00-2004_0', 'W14-1701_P10-2065_0', 'W14-3109_N12-1066_0',
    'W14-3332_P03-1021_1', 'W14-3342_N07-1051_0', 'W14-3902_N10-1027_0', 'W14-4308_N13-1127_1',
    'W14-5506_N10-1093_1', 'W15-0629_N09-1068_0', 'W15-1811_E09-1005_1', 'W15-3057_D13-1160_0',
    'W15-3206_W12-2301_1', 'W15-4108_P11-1133_0', 'W15-4634_W04-1013_0', 'W15-4703_W13-2111_1',
    'W15-5310_W13-2417_0', 'W15-5507_Q14-1019_0', 'W15-5619_P13-1010_0', 'W99-0614_W95-0103_0',
]

EXCLUIR = set(EXCLUIR_TALLER) | set(EXCLUIR_CALIBRACIONES) | set(EXCLUIR_PRUEBA_JUECES)   # con estas se fijó el orden al azar

EXCLUIR_POSTERIORES = [  # agregadas por John el 3-nov después de fijar el orden (G1.1 e ids_piloto_historico_no_test_final.txt)
    'D15-1032_P05-1012_0', 'D15-1105_W02-1020_0', 'D15-1288_P09-2086_0', 'D15-1294_P14-2085_4',
    'D15-1299_P13-2088_1', 'J15-3004_P02-1026_0', 'P14-1048_P12-1007_0', 'P14-1048_P12-1007_2',
    'P15-1020_P03-1021_0', 'P15-1029_W13-0109_0', 'P15-1086_W03-0203_0', 'P15-1089_W10-0513_0',
    'P15-1122_W11-0908_0', 'P15-1148_P10-1110_2', 'P15-1156_P14-2029_2', 'P15-1169_N13-1090_0',
    'S15-1021_N13-1090_0', 'S15-1023_P08-1028_0', 'S15-2047_S15-2044_0', 'W09-0704_E89-1009_0',
    'W15-0807_P08-1090_0', 'W15-2710_P11-1082_0', 'W15-3037_D14-1162_0', 'W15-3060_D08-1088_0',
    'W15-4314_P14-2111_4',
]

EXCLUIR_TOTAL = EXCLUIR | set(EXCLUIR_POSTERIORES)
print(len(EXCLUIR), 'citas excluidas al fijar el orden +', len(EXCLUIR_POSTERIORES), 'posteriores')

In [ ]:
# Ejemplos explicados del taller de anotación del equipo (proyecto_de_grado/anotacion/taller/):
# [función, se suele confundir con, texto en español, palabra señal, por qué es esa función]
EJEMPLOS = [
 [
  "Background",
  "Gap",
  "algoritmo no trivial de búsqueda de nombres entre idiomas en la literatura. Sin embargo, se han estudiado varias técnicas para extraer transliteraciones de nombres a partir de corpus monolingües y comparables (OTHERCIT), (TARGETCIT), (OTHERCIT), (OTHERCIT), (OTHERCIT). Estas técnicas emplean diversos modelos de similitud de transliteración. Se usaron unigramas y bigramas de caracteres como atributos para aprender un modelo discriminativo de transliteración",
  "se han estudiado varias técnicas",
  "La oración enumera técnicas ya estudiadas para minar transliteraciones; la cita objetivo es una de cinco en la lista. No se usa, no se contrasta y no se critica: solo ubica el trabajo dentro de la literatura."
 ],
 [
  "Background",
  "Gap",
  "que a partir de un corpus estándar grande. Otros investigadores (OTHERCIT), (OTHERCIT) usan técnicas de agrupamiento combinadas con atributos de dependencias sintácticas para identificar relaciones ES-UN en grandes colecciones de texto. (TARGETCIT) y (OTHERCIT) se centran en la similitud léxica de compuestos nominales no vistos con los que aparecen en el entrenamiento. Sin embargo, aunque la solución basada en la web podría superar el problema de la escasez de datos, el actual",
  "se centran en la similitud léxica",
  "Describe qué hacen los trabajos previos. El «However» que sigue critica la solución basada en web, que es otra línea de trabajo; sobre la cita objetivo solo se informa. Describir sin juzgar es Background."
 ],
 [
  "Background",
  "Identification of the Originator",
  "genera resúmenes informativos y legibles. 2. Trabajo relacionado. Se han realizado varios estudios sobre sistemas automáticos de resumen abstractivo de reuniones. Uno de ellos es el sistema propuesto por TARGETCIT. Su enfoque primero agrupa las intervenciones de las personas en comunidades (OTHERCIT) y luego construye un grafo de implicación sobre cada una para seleccionar las intervenciones más relevantes. Después aplica",
  "Uno de ellos es el sistema propuesto por",
  "Trampa de «proposed by». Está en Related Work y describe uno de varios sistemas. «proposed by» solo dice de quién es el sistema; no reclama prioridad (no hay «first» ni «introduced»). Es Background."
 ],
 [
  "Background",
  "Improvement / Modification",
  "para estimar modelos de traducción en espacio continuo (CTM) (OTHERCIT). 3.1 Modelos de traducción de n-gramas. El enfoque basado en n-gramas en traducción automática es una variante del enfoque basado en frases (TARGETCIT). Introducido en (OTHERCIT) y extendido en (Mariño et al., 2006; Crego y Mariño, 2006), este enfoque se basa en una factorización específica de la probabilidad conjunta de los pares de oraciones paralelas",
  "es una variante del enfoque basado en frases",
  "Trampa de «variant». «The n-gram-based approach… is a variant of the phrase-based approach ([CITA])» describe la relación entre dos enfoques que ya existen; el trabajo actual no modifica nada. Es contexto."
 ],
 [
  "Background",
  "",
  "interfaz con formas más tradicionales de composición (véase la Sección 3). La cuestión de la composicionalidad semántica también se discute activamente en la literatura de semántica distribucional (véase, p. ej., (OTHERCIT; TARGETCIT; OTHERCIT)), que investiga cómo combinar vectores. Esto podría verse como composición al nivel de las intensiones (si uno considera las representaciones distribucionales como intensiones, como se insinúa de varias formas",
  "véase, p. ej.,",
  "«is also actively discussed in the distributional semantics literature (see, e.g., (…; [CITA]; …))». El «see, e.g.» introduce ejemplos de que el tema se discute, no remite a detalles. Con la marca visible queda claro que la cita objetivo es una de tres en una enumeración. En la hoja del piloto esa marca no se veía."
 ],
 [
  "Background",
  "",
  "El segundo enfoque presentado en este artículo se apoya en embeddings de palabras para crear representaciones de pares de palabras. Numerosos experimentos han aprovechado los embeddings de palabras para encontrar relaciones semánticas generales (TARGETCIT; OTHERCIT). Sin embargo, solo un trabajo ha aplicado embeddings de palabras para clasificar instancias de una relación semántica léxica, en concreto la relación hiponimia-hiperonimia (OTHERCIT). Esta relación es",
  "Numerosos experimentos han aprovechado los embeddings de palabras",
  "«Extensive experiments have leveraged word embeddings to find general semantic relations ([CITA]; …)» resume una línea de trabajo. La frase siguiente («Nevertheless, only one work…») abre una brecha, pero con otra cita. Para la cita objetivo es contexto."
 ],
 [
  "Background",
  "",
  "[Restos de una figura que se colaron en el texto: «OK», «BAD», objetivo t1…t5, fuente s1…s5, |V|, concatenar, tanh(W1+b1), tanh(W2+b2)] … alinear representaciones de la fuente (Miko[lov…]; TARGETCIT, entre otros), o produciendo un desempeño del estado del arte en aplicaciones como el modelado del lenguaje (OTHERCIT, entre otros) o la traducción automática estadística (OTHERCIT, entre otros). La propiedad de que",
  "entre otros",
  "Texto con ruido de extracción (restos de una figura). Se lee que los embeddings demostraron su utilidad «…(Miko[lov]; [CITA] inter alia)». La reconciliación eligió Background, pero con la regla Background vs Evidence hay argumentos para Evidence: la cita respalda que los embeddings funcionan. Caso para la sesión de calibración."
 ],
 [
  "Background",
  "",
  "ha atraído mucha atención en la comunidad. El trabajo reciente ha cubierto varias áreas sobre eventos, como la detección de menciones de eventos (OTHERCIT), la correferencia de eventos (OTHERCIT) y la comprensión de guiones o scripts (TARGETCIT; OTHERCIT). La detección de menciones de eventos es el paso de preprocesamiento fundamental para estas tareas. Sin embargo, las investigaciones posteriores sobre eventos suelen dedicar un esfuerzo mínimo a esa detección. Por ejemplo, en",
  "El trabajo reciente ha cubierto varias áreas",
  "«Recent work has covered several areas… script understanding ([CITA]; …)» es una enumeración. La brecha aparece después («However, downstream event researches often make minimal effort…») y no habla de este trabajo."
 ],
 [
  "Background",
  "",
  "[Ecuación (8): la tasa de entropía h* definida como un límite.] En teoría, el comportamiento de la tasa de entropía con respecto al tamaño del texto ha sido controvertido. Por un lado, ha habido indicios de que la tasa de entropía es constante (TARGETCIT; OTHERCIT). Esos trabajos sostienen que la tasa de entropía del lenguaje natural podría ser constante. Sin embargo, por la dificultad de obtener el valor real de h* a partir de un texto, esos argumentos son",
  "Por un lado",
  "«On the one hand, there have been indications of entropy rate constancy ([CITA]; …)», y después viene el otro lado. La reconciliación eligió Background. Hay argumento para Evidence (la cita respalda «indications of constancy») y para Comparison si se lee como contraste entre trabajos citados. Caso para la sesión de calibración."
 ],
 [
  "Gap",
  "Background",
  "Sanfilippo y Poznanski [1992]; Manning [1993]; Collins [1997]). Otros han abordado el problema de la clasificación semántica léxica, pero usando solo frecuencias de subcategorización como datos de entrada (TARGETCIT; Schulte im Walde 2000). En concreto, estos investigadores no han abordado explícitamente la definición de atributos que capten directamente las diferencias de rol temático que no se reflejan en la subcategorización",
  "usando solo frecuencias de subcategorización",
  "La cita está en la frase que dice que esos trabajos usan «only subcategorization frequencies», y la siguiente remata: «have not explicitly addressed». La función es mostrar la limitación que justifica el estudio actual."
 ],
 [
  "Gap",
  "Background",
  "para organización y comida. … para LDA o nCRP, porque no están mapeados a WordNet. 6 Trabajo relacionado. Existe un gran volumen de trabajo previo sobre cómo extender WORDNET con conceptos e instancias adicionales (TARGETCIT; OTHERCIT); estos métodos no abordan directamente los atributos. La literatura previa sobre extracción de atributos aprovecha diversas fuentes de datos y procedimientos de extracción (OTHERCIT; Paşca y",
  "estos métodos no abordan directamente los atributos",
  "Primero reconoce el trabajo previo y en la misma oración marca su límite: «these methods do not address attributes directly». Ese límite es justo lo que el artículo va a cubrir."
 ],
 [
  "Gap",
  "Background",
  "para todos los algoritmos, donde nuestro algoritmo tuvo el mejor desempeño. También evaluamos nuestro enfoque con más rigor que otros algoritmos de resolución de metonimia. Algunos investigadores usan solo ejemplos construidos (OTHERCIT; TARGETCIT; OTHERCIT) y no reportan ningún resultado numérico. Otros (OTHERCIT) usan datos reales que, sin embargo, parecen analizados según las intuiciones subjetivas de una sola persona, y no como",
  "no reportan ningún resultado numérico",
  "La cita ilustra a los investigadores que «use constructed examples only» y «do not report any numerical results»: una debilidad metodológica que el autor dice superar con su evaluación."
 ],
 [
  "Basis",
  "Application",
  "el algoritmo PageRank en este artículo. En los últimos años se usan ampliamente muchos algoritmos para incorporar modelos de grafos de palabras e información temática en la caminata aleatoria. Nuestro trabajo está inspirado originalmente en TARGETCIT. Su método ejecuta un Topical PageRank (TPR) descompuesto para cada tema por separado y luego calcula los puntajes de las palabras respecto a los distintos temas. Al fijar los pesos de las aristas, solo la coocurrencia de palabras",
  "Nuestro trabajo está inspirado originalmente en",
  "«Our work is originally inspired by» declara una dependencia conceptual: el trabajo nuevo parte de la idea del citado. No dice que lo use tal cual (Application) ni que modifique su método (Improvement)."
 ],
 [
  "Basis",
  "Application",
  "el reto de modelar los argumentos implícitos que introducen los predicados nominales. En un estudio manual descubrimos que el 65 % de nuestros sintagmas nominales contienen relaciones implícitas. Nos apoyamos en las ideas de TARGETCIT, quienes estudiaron cómo modelar la estructura sintáctica de los sintagmas nominales en CCGBank. Aunque somos, hasta donde sabemos, los primeros en estudiar sintagmas nominales compuestos para el análisis semántico hacia bases de conocimiento",
  "Nos apoyamos en las ideas de",
  "«We build on insights from» indica que el estudio se apoya en las ideas del citado. Son ideas, no una herramienta que se ejecuta; por eso es Basis y no Application."
 ],
 [
  "Basis",
  "Improvement / Modification",
  "finalmente, reportamos la exactitud al recuperar las formas lógicas correctas zi en un subconjunto del conjunto de prueba anotado a mano. 3 Trabajo relacionado. Nuestro aprendizaje está inspirado en el enfoque de aprendizaje por refuerzo (RL) de TARGETCIT y en métodos relacionados (OTHERCIT), pero usa actualizaciones de un modelo de variables latentes dentro de un analizador semántico. OTHERCIT extendieron su enfoque de RL para modelar instrucciones de alto nivel, que corresponden a acciones implícitas",
  "está inspirado en el enfoque de aprendizaje por refuerzo (RL) de",
  "«Our learning is inspired by the RL approach of [CITA]… but uses latent variable model updates». Toma la idea como punto de partida y construye un método propio; no toma el algoritmo del citado para cambiarlo."
 ],
 [
  "Comparison",
  "Background",
  "crear el estándar de oro y, por tanto, es el recurso principal para proponer candidatos sustitutos. 4.2 UMLS. El Unified Medical Language System (UMLS) es una ontología del dominio médico. A diferencia de TARGETCIT, que usa WordNet (OTHERCIT) para generar candidatos sustitutos y también para generar atributos, nosotros usamos UMLS únicamente para generar atributos. 4.3 Google Web1T. Usamos Google Web1T para generar n-gramas",
  "A diferencia de",
  "«In contrast to [CITA], which uses WordNet…, we use UMLS» contrasta explícitamente una decisión del trabajo actual con la del citado. Contraste explícito con el trabajo propio es Comparison."
 ],
 [
  "Comparison",
  "Gap",
  "una técnica que generaliza bien entre idiomas y logra un desempeño del estado del arte en chino y alemán. En inglés, el analizador solo es superado por el analizador con reordenamiento de TARGETCIT, que tiene acceso a una variedad de atributos que un modelo generativo no puede capturar. El espacio no permite exponer todo nuestro análisis, pero, como en el caso del inglés (OTHERCIT), la",
  "solo es superado por el analizador con reordenamiento de",
  "Compara el desempeño del parser propio con el del citado: «outperformed only by the reranking parser of [CITA]». Es un contraste directo de resultados con el trabajo actual."
 ],
 [
  "Comparison",
  "Background",
  "Este sitio web es una ayuda útil para estudiar el Corán mediante anotación gramatical, y lo usan 100.000 personas al mes. A diferencia de otros esfuerzos recientes de anotación para el árabe estándar moderno (OTHERCIT; TARGETCIT), los anotadores del Quranic Treebank contrastaron sus análisis con obras históricas basadas en la gramática árabe tradicional, conocida como iʿrāb (OTHERCIT). Para ofrecer una anotación que se ajuste de cerca",
  "A diferencia de otros esfuerzos recientes de anotación",
  "«In contrast to other recent annotation efforts (…; [CITA])» contrasta el método de anotación del Quranic Treebank con el de los citados: los anotadores del corpus coránico verificaron contra obras históricas."
 ],
 [
  "Comparison",
  "Gap",
  "mejora al usar todos los datos no etiquetados con mSDA (F1 = 54,9). 4.3 Discusión. Estos resultados son muy prometedores. Aunque no son tan altos como otros resultados publicados para la polaridad de citas (TARGETCIT), hemos mostrado que se puede mejorar la clasificación de polaridad de citas aprovechando grandes cantidades de datos anotados de otros dominios y un conjunto simple de atributos.",
  "no son tan altos como otros resultados publicados",
  "Trampa de la negación. «they are not quite as high as other published results ([CITA])» no critica al citado: compara los resultados propios con los suyos. El «not» habla del trabajo actual, no de una limitación del citado."
 ],
 [
  "Application",
  "Basis",
  "Los fragmentos de texto del paso anterior pueden contener varias oraciones, así que antes de alinear oraciones debemos segmentar el texto en oraciones. Usamos el segmentador de oraciones Punkt de NLTK (TARGETCIT) para segmentar tanto oraciones como palabras en cada fragmento. Alineamiento de oraciones: para cada par de fragmentos alineados, alineamos las oraciones con el algoritmo de OTHERCIT. Filtrado de oraciones",
  "Usamos el segmentador de oraciones Punkt de NLTK",
  "«We use the Punkt sentence splitter from NLTK» es uso directo de una herramienta del citado, sin modificarla. Es el caso típico de Application."
 ],
 [
  "Application",
  "Basis",
  "representan la misma información: el equipo de los Red Sox. Por lo tanto, resúmenes distintos que usan estos fragmentos para referirse al equipo de los Red Sox no deberían considerarse muy diferentes. Usamos el modelo Pyramid (TARGETCIT) para valorar los distintos hechos del resumen. Intuitivamente, los hechos que se mencionan con más frecuencia son los aspectos más destacados de la historia. Por eso, nuestro modelo pirámide usa la frecuencia normalizada con la que",
  "Usamos el modelo Pyramid",
  "«We use the Pyramid model ([CITA]) to value different summary factoids» aplica el método del citado tal como está para una tarea del trabajo actual."
 ],
 [
  "Application",
  "Background",
  "modelo de traducción basado en modelos IBM bidireccionales (OTHERCIT) y modelos HMM (OTHERCIT), combinados con la heurística grow-diag-final-and. Esto se calcula con la herramienta GIZA++ (OTHERCIT) y el kit de herramientas Moses (TARGETCIT). El modelo de lenguaje fue un 5-grama de SRILM (OTHERCIT). Los parámetros del sistema final se fijaron con Minimum Error-Rate Training (OTHERCIT), y la calidad de la traducción se evaluó con",
  "se calcula con la herramienta GIZA++",
  "El modelo «is computed with the GIZA++ tool and the Moses toolkit ([CITA])»: uso directo de software del citado."
 ],
 [
  "Application",
  "Identification of the Originator",
  "MT05 y MT06. La calidad de la traducción se evalúa con la métrica BLEU, versión NIST, sin distinguir mayúsculas. La prueba de significancia estadística se realiza usando el método de remuestreo bootstrap propuesto por TARGETCIT. La búsqueda en haz y la poda por cubos (OTHERCIT) se usan para reducir el espacio de búsqueda en los tres sistemas base. Por defecto, tanto el tamaño del haz como el de la lista n-best se fijan en 20.",
  "usando el método de remuestreo bootstrap propuesto por",
  "Trampa de «proposed by». El verbo que manda es «using»: el autor aplica el método del citado para su prueba de significancia. «proposed by» es solo atribución."
 ],
 [
  "Application",
  "Basis",
  "los estados resultantes para el siguiente paso. En la práctica, nuestro algoritmo de programación dinámica también corre sobre una búsqueda en haz. Para entrenar el modelo usamos el algoritmo de perceptrón promediado (OTHERCIT). Siguiendo a TARGETCIT, también usamos la estrategia de «actualización temprana» (early-update), en la que se actualiza cada vez que la secuencia de acciones correcta se cae del haz, y se descarta el resto de la secuencia.",
  "también usamos la estrategia de «actualización temprana»",
  "Trampa de «following». «Following [CITA] we also use the early-update strategy» suena a Basis, pero lo que se toma es una técnica concreta que se usa tal cual. Uso directo es Application."
 ],
 [
  "Application",
  "",
  "atributos, que llamamos CONTEXT-BASED porque se extraen del contexto de cada ocurrencia del verbo que clasificamos. Este conjunto de atributos reúne los atributos propuestos por OTHERCIT y los que propuso TARGETCIT. Además, usamos los atributos modal y negación. Extraemos estos atributos de análisis de dependencias sintácticas generados con el analizador de Stanford (OTHERCIT). El tiempo verbal y la voz se extraen siguiendo",
  "los que propuso",
  "«This feature set comprises… the ones proposed by [CITA]»: se incorporan las características tal como las definió el citado. Agregar otras propias después («In addition, we use…») no modifica las del citado."
 ],
 [
  "Improvement / Modification",
  "Application",
  "distancias mayores que k reciben la misma probabilidad. Si aun después de la interpolación la probabilidad es 0, se usa un valor muy pequeño. Esta es una versión modificada del suavizado por retroceso (backed-off) que usa TARGETCIT para aliviar los problemas de escasez de datos. Se usa una interpolación similar para el primer componente del lado derecho de la Ecuación 3, quitando a la vez la información contextual del núcleo y del dependiente",
  "Esta es una versión modificada",
  "«This is a modified version of the backed-off smoothing used by [CITA]». La palabra clave es «modified»: se toma el método del citado y se cambia."
 ],
 [
  "Improvement / Modification",
  "Application",
  "[datos editoriales de la revista: 2013, páginas 49–62, enviado 11/2012, publicado 3/2013]. Con ese fin, adaptamos el algoritmo Perceptron sensible a la pérdida (OTHERCIT; TARGETCIT) para que use una función de validación e inferencia de grueso a fino en la inducción léxica. La naturaleza conjunta de este enfoque aporta beneficios cruciales, porque permite que pistas del entorno, como el conjunto de objetos visibles",
  "adaptamos el algoritmo Perceptron sensible a la pérdida",
  "«We adapt the loss-sensitive Perceptron algorithm (…; [CITA]) to use a validation function and coarse-to-fine inference»: adapta el algoritmo agregándole componentes nuevos."
 ],
 [
  "Improvement / Modification",
  "Application",
  "respuesta a preguntas (OTHERCIT) y ha mostrado mejores resultados que los enfoques de coincidencia léxica (por ejemplo, los modelos de alineamiento de palabras). En este trabajo adaptamos un marco de red neuronal convolucional (CNN) (OTHERCIT; TARGETCIT) a este problema de emparejamiento. La arquitectura de la red se ilustra en la Fig. 6. El modelo CNN primero aplica una técnica de hashing de palabras (OTHERCIT) que descompone cada palabra en un vector de trigramas de letras",
  "adaptamos un marco de red neuronal convolucional (CNN)",
  "«We adapt a CNN framework (…; [CITA]) to this matching problem» lleva el método a un problema distinto, lo que exige cambiarlo. Adaptar es Improvement, usar tal cual es Application."
 ],
 [
  "Improvement / Modification",
  "",
  "variedades de acciones shift y reduce. Esta modificación también facilita aplicar al análisis de constituyentes un algoritmo desarrollado para el análisis de dependencias, como la programación dinámica con búsqueda en haz (TARGETCIT), que no se había aplicado al análisis de constituyentes hasta hace muy poco (OTHERCIT) (véase la Sección 7). Algoritmo 1: búsqueda BFS para análisis de constituyentes; solo se muestran las diferencias con OTHERCIT",
  "Esta modificación también facilita aplicar",
  "Se toma un algoritmo de dependency parsing ([CITA]) y se lleva a constituent parsing gracias a «this modification». Cambiar de dominio exige adaptarlo, por eso es Improvement / Modification y no Application."
 ],
 [
  "Evidence",
  "Background",
  "los atributos prosódicos capturan cómo se dice algo, pero los que representan qué se dice también son importantes. Se ha demostrado que la información léxica mejora la predicción de emociones a partir del habla en otros dominios (TARGETCIT; OTHERCIT), así que nuestro primer atributo no prosódico representa la transcripción de cada turno del estudiante como un vector de ocurrencia de palabras (que indica qué palabras aparecen en el turno).",
  "Se ha demostrado que la información léxica mejora",
  "«Lexical information has been shown to improve… ([CITA])» y luego «so our first feature…». La cita es la prueba que justifica una decisión de diseño concreta."
 ],
 [
  "Evidence",
  "Gap",
  "que esto conduce a una mejor calidad de traducción. Idealmente, produciríamos todas las segmentaciones y alineamientos posibles durante el entrenamiento. Sin embargo, se ha demostrado que esto es inviable con datos reales (TARGETCIT). Como el entrenamiento usa una versión modificada del decodificador de traducción, es sencillo aplicar poda como en la decodificación normal. Además, consideramos tres formas de aproximar la búsqueda completa",
  "se ha demostrado que esto es inviable",
  "«This has been shown to be infeasible for real-world data ([CITA])» sostiene una afirmación concreta (que es inviable) con un resultado del citado. No critica al citado: lo usa como respaldo."
 ],
 [
  "Evidence",
  "Background",
  "las formas directamente, incluso cuando hay evidencia de una traducción directa en los datos paralelos. La morfología no supervisada ha empezado a tener un papel en la traducción entre lenguas morfológicamente complejas. TARGETCIT muestran que un enfoque no supervisado de segmentación del árabe funciona tan bien como un segmentador supervisado para preprocesar el lado fuente (medido por la calidad de la traducción al inglés). Para traducir hacia lenguas morfológicamente complejas",
  "muestran que un enfoque no supervisado",
  "La afirmación es «unsupervised morphology has begun to play a role»; la frase siguiente la respalda con lo que el citado mostró: «show that an unsupervised approach… performs as well as a supervised segmenter»."
 ],
 [
  "Evidence",
  "Further Reading",
  "de la calidad de traducción cuando se apoya en conocimiento lingüístico profundo. 1 Introducción. La traducción automática estadística (SMT) es el enfoque actual del estado del arte en traducción automática; véase p. ej. TARGETCIT. Sin embargo, sus salidas siguen siendo, por lo general, bastante peores que las traducciones humanas, con diversos tipos de errores (OTHERCIT), tanto en la elección de palabras como en la gramática. Como han mostrado muchas investigaciones",
  "véase p. ej.",
  "Trampa de «see». «SMT is the current state-of-the-art approach – see e.g. [CITA]»: el «see e.g.» acompaña una afirmación central y la cita la respalda. La guía dice que «see» no basta para Further Reading."
 ],
 [
  "Evidence",
  "Comparison",
  "la distribución previa usada en la verosimilitud marginal, p(DIC), y Q, que determina la contribución de pMRF(C) a la Ec. (1). 4 Experimentos. Se ha demostrado que los sistemas de SMT basados en frases superan a los enfoques basados en palabras (TARGETCIT). Evaluamos el efecto de elegir el modelo de léxico sobre la calidad de la traducción, con dos aplicaciones dentro de un sistema de SMT basado en frases. 4.1 Aplicaciones a SMT basada en frases",
  "Se ha demostrado que los sistemas de SMT basados en frases superan",
  "Trampa de «outperform». La comparación es entre enfoques ajenos (phrase-based contra word-based), no con el trabajo actual. La cita sirve de prueba para justificar que el autor use phrase-based SMT."
 ],
 [
  "Identification of the Originator",
  "Background",
  "con variación flexiva añadida) forman nuestra lista de negadores. 4 SMT aumentada con paráfrasis. Aumentar el lado fuente de las tablas de frases de SMT con paráfrasis de los elementos fuera del vocabulario (OOV) fue introducido por TARGETCIT, y fue adoptado prácticamente «tal cual» en trabajos posteriores (OTHERCIT). Dada una frase fuente OOV f, si el modelo de traducción tiene una regla (f′, e) cuyo lado fuente f′ es una paráfrasis de f, entonces",
  "fue introducido por",
  "«…was introduced by [CITA], and was adopted practically as-is in consequent work». Atribuye explícitamente el origen de la técnica y la distingue de quienes la adoptaron después."
 ],
 [
  "Identification of the Originator",
  "Background",
  "que podemos elegir entre muchas métricas tradicionales para evaluar con nuestras nuevas referencias, y a la vez eliminar algunas deficiencias de esas métricas. El parafraseo dirigido para evaluar traducción automática fue introducido por TARGETCIT. Su algoritmo crea nuevas oraciones de referencia sustituyendo una palabra, con base en la sinonimia de WordNet (OTHERCIT) y en una evaluación del contexto. Esta solución no se puede aplicar directamente al checo",
  "fue introducido por",
  "«Targeted paraphrasing for MT evaluation was introduced by [CITA]». La oración existe para decir quién creó la idea."
 ],
 [
  "Identification of the Originator",
  "Application",
  "los sintagmas nominales base. El primer trabajo sobre este tema se hizo en los años ochenta (OTHERCIT). El conjunto de datos que se volvió estándar para evaluar enfoques de aprendizaje automático es el que usó por primera vez TARGETCIT. Está formado por los mismos segmentos de entrenamiento y prueba del Penn Treebank que la tarea de chunking (secciones 15-18 y sección 20, respectivamente). Sin embargo, como los conjuntos de datos se generaron con",
  "el que usó por primera vez",
  "«The data set that has become standard… is the one first used by [CITA]». «first used» establece prioridad: reconoce quién lo usó primero."
 ],
 [
  "Further Reading",
  "Background",
  "o PXY para P5Y) y la granularidad de la expresión temporal (por ejemplo, DAY-OF-MONTH para viernes, MONTH-OF-YEAR para febrero, etc.); [nota al pie 1] Para más detalles sobre el modelo determinista de análisis temporal, remitimos al lector a (TARGETCIT). [Sigue una tabla de resultados con columnas Entrenamiento, Prueba, P, R y F1 para TimeBank, AQUAINT y Silver.]",
  "Para más detalles",
  "«For further details… we refer the reader to ([CITA])» es la forma canónica: remite a detalles que el texto actual no desarrolla."
 ],
 [
  "Further Reading",
  "Evidence",
  "se pierde el conocimiento de que los cuatro caracteres transmiten un significado específico distinto del de cada carácter, y no siempre puede recuperarse del todo ni siquiera usando frases en sistemas de SMT basados en frases (véase TARGETCIT para más detalle). OTHERCIT y Paul et al. (2010) señalaron además que segmentar la oración fuente en unidades más grandes sí ayuda a capturar más contexto en la traducción. Por eso, en lugar de usar caracteres",
  "para más detalle",
  "«(see [CITA] for detail)»: la cita va entre paréntesis y solo ofrece más detalle sobre un punto que ya se afirmó."
 ],
 [
  "Further Reading",
  "Background",
  "se han definido para el caso en que la partición clave tiene el mismo conjunto de sintagmas nominales que la partición de respuesta. Para aplicar estas métricas a sintagmas extraídos automáticamente, se han propuesto distintos métodos (véase TARGETCIT y OTHERCIT). Como la correferencia es una tarea de agrupamiento, cualquier método general para evaluar una partición de respuesta frente a una partición clave (por ejemplo, Kappa (OTHERCIT)) puede usarse para puntuar la correferencia",
  "se han propuesto distintos métodos (véase",
  "«different methods have been proposed (see [CITA] and …)» remite a dónde encontrar esos métodos sin describirlos. Si el texto los describiera, sería Background."
 ],
 [
  "Further Reading",
  "",
  "exactitud de clasificación. [Tabla de equipos participantes] Equipo · Afiliación y referencia: Al-Bayan, Universidad de Alejandría, Egipto (OTHERCIT) · CICBUAPnlp, Instituto Politécnico Nacional, México · CoMiC, Universidad de Tubinga, Alemania (TARGETCIT) · ECNU, Universidad Normal de China Oriental (OTHERCIT) · FBK-HLT, Fondazione Bruno Kessler, Italia (OTHERCIT) · HITSZ-ICRC, Instituto de Tecnología de Harbin, China (OTHERCIT) · ICRC-HIT, Instituto de Tecnología de Harbin",
  "",
  "Es una tabla de equipos participantes, no prosa: la cita indica dónde leer la descripción del sistema CoMiC y no respalda ninguna afirmación. Propuesta para el equipo: acordar si los contextos que vienen de tablas se excluyen del dataset."
 ]
]

In [ ]:
def leer_csv(ruta):
    """Lee un CSV de Excel (con BOM) como lista de diccionarios."""
    with open(ruta, encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

jurado = {f['context_id']: f for f in leer_csv(JURADO)}                # voto y partición de cada cita
citas = {f['context_id']: f for f in leer_csv(CITAS)}                  # textos de cada cita
assert len(jurado) == 18000 and set(jurado) <= set(citas), 'Los archivos no corresponden'
print('Votos del jurado (todas las particiones):', Counter(f['voto'] for f in jurado.values()).most_common())

## Paso 2. Calibración: 20 citas de 2014

Dos citas por clase cuando hay (Identification of the Originator tiene muy pocas) y el resto "sin consenso", que son los casos difíciles: justo lo que más sirve discutir. Salen de la partición val (2014) para no gastar citas del test.

In [ ]:
def muestrear_calibracion(jurado, rng):
    """2 citas por clase de la partición val y el resto sin consenso, hasta N_CALIBRACION."""
    val = [f for f in jurado.values() if f['particion'] == 'val' and f['context_id'] not in EXCLUIR]
    elegidas = []
    for c in CLASES:
        candidatas = sorted(f['context_id'] for f in val if f['voto'] == c)   # ordenar antes de muestrear: reproducible
        elegidas += rng.sample(candidatas, min(2, len(candidatas)))
    dificiles = sorted(f['context_id'] for f in val if f['voto'] == 'SIN_CONSENSO')
    elegidas += rng.sample(dificiles, N_CALIBRACION - len(elegidas))
    rng.shuffle(elegidas)                                              # mezclar para que no vengan agrupadas por clase
    return elegidas

rng = random.Random(SEMILLA)
calibracion = muestrear_calibracion(jurado, rng)
print(len(calibracion), 'citas de calibración:', Counter(jurado[i]['voto'] for i in calibracion))

## Paso 3. Tandas del Test Gold: citas de 2015 por clase y por pareja

Para cada clase se ordenan al azar, una sola vez, todas las citas de 2015 disponibles. La tanda 1 toma las primeras hasta la cuota del tope 500; la tanda 2, las siguientes hasta la cuota del tope 1.000; la tanda 3, hasta la del tope 2.000. Así la suma de las tandas hechas siempre es una muestra al azar, y las tandas futuras quedan fijadas desde hoy.

La cuota de cada clase es el 15 % de lo que tendría el dataset con ese tope, sin pasar de lo que hay ni de lo que cabe en la partición test. Cada cita va a una pareja; con 5 personas hay 10 parejas y cada persona queda en 4.

**Exclusiones posteriores:** el 3-nov, después de repartir la tanda 1, John agregó 25 citas que no deben ir al Test Gold (5 usadas para diseñar la alineación G1.1 y 20 del piloto histórico). Para no cambiar lo ya repartido, el orden al azar se mantiene: en la tanda 1, cada cita excluida se reemplaza **en su misma posición** por la siguiente de su clase (así conserva la pareja y la fila de la hoja), y las tandas 2 y 3 se arman con lo que queda.

In [ ]:
def cuotas_test(jurado, excluir, tope):
    """Tamaño del dataset y del Test Gold por clase con un tope dado."""
    total = Counter(f['voto'] for f in jurado.values())                  # votos en todas las particiones
    disponibles = defaultdict(list)                                      # citas de 2015 que pueden entrar al test
    for f in jurado.values():
        if f['particion'] == 'test' and f['voto'] in CLASES and f['context_id'] not in excluir:
            disponibles[f['voto']].append(f['context_id'])
    dataset, cuota = {}, {}
    for c in CLASES:
        dataset[c] = min(tope, total[c], math.floor(len(disponibles[c]) / FRACCION_TEST))   # el 15 % debe caber en test
        cuota[c] = min(round(dataset[c] * FRACCION_TEST), len(disponibles[c]))
    return dataset, cuota, disponibles

def asignar_parejas(ids, rng):
    """Reparte las citas en las 10 parejas posibles, mezcladas, para que la carga quede pareja."""
    parejas = list(combinations(PERSONAS, 2))
    ids = list(ids)
    rng.shuffle(ids)
    return {cid: parejas[k % len(parejas)] for k, cid in enumerate(ids)}

excluir_test = EXCLUIR | set(calibracion)
cuotas = {k: cuotas_test(jurado, excluir_test, tope) for k, tope in TOPES.items()}   # (dataset, cuota, disponibles) por tanda
disponibles = cuotas[1][2]
orden = {c: rng.sample(sorted(disponibles[c]), len(disponibles[c])) for c in CLASES}   # orden al azar, una vez por clase
posteriores = set(EXCLUIR_POSTERIORES)
assert not set(calibracion) & posteriores, 'Una exclusión posterior cae en la calibración'
tandas = {k: [] for k in TOPES}
for c in CLASES:
    cola = [cid for cid in orden[c] if cid not in posteriores]       # el mismo orden, sin las exclusiones posteriores
    base1 = orden[c][:cuotas[1][1][c]]                               # la tanda 1 tal como se repartió
    reemplazos = iter(cid for cid in cola if cid not in base1)       # las siguientes de la clase, en orden
    t1 = [cid if cid not in posteriores else next(reemplazos, None) for cid in base1]   # reemplazo en la misma posición
    t1 = [cid for cid in t1 if cid]
    resto = [cid for cid in cola if cid not in t1]
    n2 = cuotas[2][1][c] - cuotas[1][1][c]
    n3 = cuotas[3][1][c] - cuotas[2][1][c]
    tandas[1] += t1
    tandas[2] += resto[:n2]
    tandas[3] += resto[n2:n2 + n3]
pareja_de, tanda_de = {}, {}
for k in TOPES:
    pareja_de.update(asignar_parejas(tandas[k], random.Random(f'{SEMILLA}-tanda{k}')))
    tanda_de.update({cid: k for cid in tandas[k]})

print(f'{"clase":34s} {"disp. 2015":>10s}' + ''.join(f'{"test tope " + str(t):>14s}' for t in TOPES.values()))
for c in CLASES:
    print(f'{c:34s} {len(disponibles[c]):10d}' + ''.join(f'{cuotas[k][1][c]:14d}' for k in TOPES))
print(f'{"TOTAL acumulado":34s} {"":10s}' + ''.join(f'{sum(cuotas[k][1].values()):14d}' for k in TOPES))
print(f'{"Dataset con ese tope":34s} {"":10s}' + ''.join(f'{sum(cuotas[k][0].values()):14d}' for k in TOPES))
for k in TOPES:
    carga = Counter(p for cid in tandas[k] for p in pareja_de[cid])
    print(f'Tanda {k}: {len(tandas[k])} citas | por persona: {dict(carga)}')

## Paso 4. Traducción de apoyo al español (GPT)

Traduce contexto, título y resumen de la calibración y de **todas las tandas** (unas 1.560 citas), para correr una sola vez. Revisa que las marcas `TARGETCIT` y `OTHERCIT` queden iguales; si no, reintenta una vez y si sigue mal la marca para revisar. Manda varias citas a la vez (`HILOS`) para que tarde menos. Se guarda en Drive y se puede reanudar: si se corta o alguna cita falla, se repite la celda. Necesita `OPENAI_API_KEY` en los Secretos de Colab. Si `TRADUCIR = False`, las hojas salen sin traducción.

Costo estimado con `gpt-6.1-sol` (US$2 y US$10 por millón de tokens de entrada y salida): del orden de US$10 para las ~1.560 citas, casi todo por el texto de salida (no medido). Con `gpt-6.1-terra` (US$1 y US$5) sería la mitad.

In [ ]:
TRADUCIR = True
MODELO_TRAD = 'gpt-6.1-sol'                                          # mismo modelo de la referencia (verificado el 3-nov-2026)
INSTRUCCION = ('Traduce al español el siguiente texto académico de lingüística computacional. '
               'Conserva exactamente las marcas TARGETCIT y OTHERCIT, sin traducirlas ni moverlas. '
               'No agregues comentarios ni explicaciones: responde solo con la traducción.')
ruta_trad = SALIDA / 'traducciones.jsonl'

def leer_jsonl(ruta):
    """Registros ya guardados, por cita, para poder reanudar."""
    if not ruta.exists():
        return {}
    return {d['context_id']: d for d in (json.loads(l) for l in ruta.read_text(encoding='utf-8').splitlines() if l.strip())}

def marcas_iguales(original, traduccion):
    """True si la traducción conserva cuántas marcas TARGETCIT y OTHERCIT tiene el original."""
    return all(original.count(m) == traduccion.count(m) for m in ('TARGETCIT', 'OTHERCIT'))

def recortar(texto, maximo):
    return texto if len(texto) <= maximo else texto[:maximo].rsplit(' ', 1)[0] + ' …'

HILOS = 8                 # citas que se traducen al mismo tiempo; si aparecen errores 429 (límite de la API), bájalo a 4

def registro_traducido(cid, traducir):
    """Traduce contexto, título y resumen de una cita y arma su registro."""
    f = citas[cid]
    ctx = f['Contexto TARGETCIT']
    ctx_es = traducir(ctx)
    if not marcas_iguales(ctx, ctx_es):
        ctx_es = traducir(ctx)                                       # un reintento si se perdieron marcas
    return {'context_id': cid, 'contexto_es': ctx_es, 'revisar': not marcas_iguales(ctx, ctx_es),
            'titulo_es': traducir(f['Título citado']) if f['Título citado'].strip() else '',
            'resumen_es': traducir(recortar(f['Resumen citado'], RESUMEN_MAX)) if f['Resumen citado'].strip() else '',
            'modelo': MODELO_TRAD}

def traducir_pendientes(ids, traducir, traducciones, ruta, hilos):
    """Traduce en paralelo las citas que faltan y guarda cada una en cuanto termina (reanudable)."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    pendientes = [i for i in ids if i not in traducciones]
    print(len(pendientes), 'citas por traducir')
    fallas = []
    with open(ruta, 'a', encoding='utf-8') as salida, ThreadPoolExecutor(max_workers=hilos) as pool:
        futuros = {pool.submit(registro_traducido, cid, traducir): cid for cid in pendientes}
        for k, fut in enumerate(as_completed(futuros), 1):
            cid = futuros[fut]
            try:
                registro = fut.result()
            except Exception as e:                                   # una cita que falla no detiene las demás
                fallas.append((cid, f'{type(e).__name__}: {e}'))
                continue
            salida.write(json.dumps(registro, ensure_ascii=False) + '\n')   # solo el hilo principal escribe
            salida.flush()
            traducciones[cid] = registro
            if k % 100 == 0:
                print(f'  {k}/{len(pendientes)}')
    if fallas:
        print(len(fallas), 'citas fallaron (repite la celda para reintentarlas). Ejemplo:', fallas[0])
    return fallas

todas = calibracion + [cid for k in TOPES for cid in tandas[k]]       # calibración y las tres tandas
traducciones = leer_jsonl(ruta_trad)
if TRADUCIR:
    from openai import OpenAI
    from google.colab import userdata
    cliente = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))          # la llave se lee del secreto
    def traducir(texto):
        r = cliente.responses.create(model=MODELO_TRAD, instructions=INSTRUCCION, input=texto, max_output_tokens=4096)
        return (r.output_text or '').strip()
    t0 = time.time()
    traducir_pendientes(todas, traducir, traducciones, ruta_trad, HILOS)
    print(f'Tiempo: {(time.time() - t0) / 60:.0f} min')
print('Traducidas:', len(traducciones), 'de', len(todas), '| con marcas para revisar:', sum(d['revisar'] for d in traducciones.values()))

## Paso 5. Hojas de Excel por anotador y clave del coordinador

Cada persona recibe dos archivos: `Calibracion3_<Apellido>.xlsx` (las mismas 20 citas para todos) y `TestGold_T1_v2_<Apellido>.xlsx` (sus citas de la tanda 1). La versión va en el nombre (`VERSION_HOJAS`) para que nadie trabaje sobre una copia vieja. Formato igual al de las calibraciones anteriores: hoja `Mis_anotaciones` con listas desplegables y la columna `Estado` que se calcula sola, `Control` con el avance, `Guia_rapida` y `9_funciones` (definición, frases típicas y con qué no confundir cada función), más `Ejemplos_explicados` con los 41 casos del taller (en español, con la palabra señal y por qué es esa función). Ninguno de esos ejemplos está en la calibración ni en el test.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

ENCABEZADOS = ['N°', 'context_id', 'Contexto original (inglés)', 'Contexto en español (ayuda)', 'Título citado',
               'Título en español', 'Resumen citado', 'Resumen en español', '¿Evaluable?', 'Función principal (una)',
               'Confianza', 'Justificación / duda', 'Estado']
ANCHOS = [5, 22, 60, 60, 30, 30, 50, 50, 14, 26, 11, 40, 13]

def formula_estado(r):
    """Estado de la fila, como en la campaña de Calibración 2: se calcula solo en Excel. Toda fila pide justificación."""
    return (f'=IF(I{r}="","PENDIENTE",IF(I{r}="No evaluable",IF(L{r}="","REVISAR","REGISTRADO"),'
            f'IF(I{r}="Ambiguo",IF(OR(J{r}="",L{r}=""),"INCOMPLETO","COMPLETO"),'
            f'IF(AND(J{r}<>"",K{r}<>"",L{r}<>""),"COMPLETO","INCOMPLETO"))))')
GUIA = [
    'Decide con el texto ORIGINAL en inglés. La traducción es solo una ayuda y puede tener errores.',
    'TARGETCIT es la cita que debes clasificar. OTHERCIT son otras citas del mismo párrafo.',
    'En ¿Evaluable? elige Evaluable, Ambiguo (dos lecturas igual de posibles) o No evaluable (el texto no deja ver la función).',
    'Si es Evaluable, elige UNA de las nueve funciones y la confianza.',
    'Si es Ambiguo, elige igual la función más probable y explica la duda en Justificación: la guía del equipo pide una sola etiqueta por caso.',
    'Si es No evaluable, deja la función vacía y explica por qué en Justificación.',
    'La columna Estado se llena sola (PENDIENTE, COMPLETO, INCOMPLETO, REGISTRADO o REVISAR). Al terminar no debe quedar ningún PENDIENTE ni INCOMPLETO.',
    'Confianza: Alta, Media o Baja según qué tan seguro estés.',
    'Escribe una frase en Justificación: qué palabras o idea del texto te hicieron decidir.',
    'Trabaja solo. No mires las respuestas de tu pareja ni de ningún modelo (QLoRA, Qwen, Llama, GPT, Claude).',
    'No insertes, borres ni reordenes filas.',
    'Reglas de desempate: una lista de trabajos previos suele ser Background; Comparison solo con contraste explícito.',
    'Background ubica el trabajo en la literatura; Evidence respalda una afirmación concreta.',
    'Basis es una dependencia conceptual; Application es uso práctico directo de un método, herramienta o dato.',
    'Si el método citado se usa tal cual: Application. Si se adapta o modifica: Improvement / Modification.',
    'Identification of the Originator exige atribución explícita del origen ("introduced", "first proposed"), no solo una cita antigua.',
    'Further Reading: la cita remite a más detalles; no basta con que aparezca "see".',
    'Antes de empezar, lee la hoja 9_funciones y los ejemplos de la hoja Ejemplos_explicados (casos del taller, que no están en esta tanda).',
]
DEFINICIONES = [  # (función, qué es, cómo reconocerla, no confundir con)
    ('Background', 'Contexto general: resume literatura previa, el estado de un área o antecedentes históricos.',
     'Listas de trabajos previos; "has been studied", "previous work", "several approaches".',
     'Evidence (respalda una afirmación concreta) y Comparison (exige contraste explícito).'),
    ('Gap', 'Señala algo que no se ha hecho, una limitación de trabajos previos o una brecha que justifica el estudio.',
     '"however", "little attention", "remains unexplored", "fail to", "do not address".',
     'Background (solo describe, no señala una carencia).'),
    ('Basis', 'El trabajo citado es la base intelectual o conceptual directa; la propuesta se construye sobre él.',
     '"we build on", "following the framework of", "based on the theory of".',
     'Application (uso práctico de una herramienta o dato, no una base conceptual).'),
    ('Comparison', 'Compara explícitamente métodos, resultados, datos o enfoques con el trabajo citado o entre citados.',
     '"unlike", "in contrast to", "outperforms", "similar to", "compared with".',
     'Background (una lista de trabajos sin contraste no es comparación).'),
    ('Application', 'Usa directamente un método, algoritmo, herramienta, dataset o procedimiento del citado, sin modificarlo.',
     '"we use", "we employ", "using the parser of", "trained on the corpus of".',
     'Improvement / Modification (si lo adapta) y Basis (si es solo inspiración conceptual).'),
    ('Improvement / Modification', 'Adapta, amplía, modifica o mejora un método o herramienta del citado.',
     '"we extend", "we modify", "adapted from", "a variant of", "we improve".',
     'Application (si se usa tal cual).'),
    ('Evidence', 'Respalda una afirmación, hipótesis, decisión metodológica, dato o interpretación concreta.',
     '"it has been shown that", "as shown by", "consistent with", "confirms".',
     'Background (si la oración seguiría siendo una descripción general del campo).'),
    ('Identification of the Originator', 'Reconoce a quien introdujo originalmente una idea, método o teoría.',
     '"introduced by", "first proposed", "originally developed", "coined".',
     'Background (una cita antigua no basta: debe atribuir el origen).'),
    ('Further Reading', 'Remite al lector a una fuente con detalles adicionales que el texto no desarrolla.',
     '"see … for details", "for a review, see", "described in detail in".',
     'Evidence (no basta con que aparezca "see": debe remitir a más detalles).'),
]

def hoja_anotador(ruta, titulo, nota, ids):
    """Crea el Excel de un anotador con sus citas, listas desplegables y las hojas de ayuda."""
    wb = Workbook()
    ws = wb.active
    ws.title = 'Mis_anotaciones'
    ws['A1'] = titulo
    ws['A1'].font = Font(bold=True, size=13)
    ws['A3'] = nota
    for col, (enc, ancho) in enumerate(zip(ENCABEZADOS, ANCHOS), 1):
        celda = ws.cell(row=5, column=col, value=enc)
        celda.font = Font(bold=True)
        celda.fill = PatternFill('solid', fgColor='DDE7F3')
        ws.column_dimensions[celda.column_letter].width = ancho
    for n, cid in enumerate(ids, 1):
        f, t = citas[cid], traducciones.get(cid, {})
        fila = [n, cid, f['Contexto TARGETCIT'], t.get('contexto_es', ''), f['Título citado'], t.get('titulo_es', ''),
                recortar(f['Resumen citado'], RESUMEN_MAX), t.get('resumen_es', ''), '', '', '', '', formula_estado(5 + n)]
        for col, valor in enumerate(fila, 1):
            ws.cell(row=5 + n, column=col, value=valor).alignment = Alignment(wrap_text=True, vertical='top')
    ultima = 5 + len(ids)
    c = wb.create_sheet('Control')                                    # avance del anotador, como en Calibración 2
    c.append(['AVANCE', 'Valor', 'Descripción'])
    c.append(['Citas asignadas', f'=COUNTA(Mis_anotaciones!B6:B{ultima})', 'Filas de esta hoja'])
    for estado, desc in (('COMPLETO', 'Con función y justificación (y confianza si es Evaluable)'), ('REGISTRADO', 'No evaluables con justificación'),
                         ('PENDIENTE', 'Sin empezar'), ('INCOMPLETO', 'Les falta función, confianza o justificación'),
                         ('REVISAR', 'No evaluables sin justificación')):
        c.append([estado.capitalize(), f'=COUNTIF(Mis_anotaciones!M6:M{ultima},"{estado}")', desc])
    c.column_dimensions['A'].width = 20
    c.column_dimensions['C'].width = 50
    for col, opciones in (('I', EVALUABLE), ('J', CLASES), ('K', ['Alta', 'Media', 'Baja'])):
        dv = DataValidation(type='list', formula1='"' + ','.join(opciones) + '"', allow_blank=True)
        dv.add(f'{col}6:{col}{ultima}')
        ws.add_data_validation(dv)
    ws.freeze_panes = 'C6'                                            # deja visibles el número, el id y los encabezados
    g = wb.create_sheet('Guia_rapida')
    g.append(['Paso', 'Instrucciones para el anotador'])
    for k, texto in enumerate(GUIA, 1):
        g.append([k, texto])
    g.column_dimensions['B'].width = 120
    d = wb.create_sheet('9_funciones')
    d.append(['Función', 'Qué es (guía de anotación)', 'Cómo reconocerla (frases típicas, orientativas)', 'No confundir con'])
    for fila in DEFINICIONES:
        d.append(list(fila))
    for col, ancho in zip('ABCD', (32, 70, 60, 60)):
        d.column_dimensions[col].width = ancho
    e = wb.create_sheet('Ejemplos_explicados')
    e.append(['Función', 'Se suele confundir con', 'Texto (en español; TARGETCIT es la cita)', 'Palabra señal', 'Por qué es esa función'])
    for fila in EJEMPLOS:
        e.append(fila)
    for col, ancho in zip('ABCDE', (30, 26, 80, 26, 80)):
        e.column_dimensions[col].width = ancho
    for hoja in (c, g, d, e):                                         # encabezado en negrita y texto ajustado
        for celda in hoja[1]:
            celda.font = Font(bold=True)
            celda.fill = PatternFill('solid', fgColor='DDE7F3')
        for fila in hoja.iter_rows(min_row=2):
            for celda in fila:
                celda.alignment = Alignment(wrap_text=True, vertical='top')
    wb.save(ruta)

for p in PERSONAS:
    hoja_anotador(SALIDA / f'Calibracion3_{p}.xlsx', f'CALIBRACIÓN 3 DE FUNCIONES DE CITA  |  {p.upper()}',
                  'Las mismas 20 citas para las 5 personas (partición val, 2014). Sirve para alinear criterios; no es Test Gold.',
                  calibracion)
    mias = [cid for cid in tandas[TANDA_ACTUAL] if p in pareja_de[cid]]
    rng_p = random.Random(f'{SEMILLA}-{p}-tanda{TANDA_ACTUAL}')      # orden propio de cada persona, reproducible
    rng_p.shuffle(mias)
    hoja_anotador(SALIDA / f'TestGold_T{TANDA_ACTUAL}_{VERSION_HOJAS}_{p}.xlsx', f'TEST GOLD, TANDA {TANDA_ACTUAL}  |  {p.upper()}',
                  'Citas de 2015 (partición test). Cada cita la anota otra persona por separado. Anota a ciegas y sin consultar modelos.',
                  mias)

with open(SALIDA / f'clave_coordinador_{VERSION_HOJAS}.csv', 'w', encoding='utf-8-sig', newline='') as f:   # NO compartir con anotadores
    w = csv.writer(f)
    w.writerow(['context_id', 'grupo', 'tanda', 'particion', 'voto_jurado', 'jueces_de_acuerdo', 'anotador_1', 'anotador_2'])
    for cid in calibracion:
        w.writerow([cid, 'calibracion3', '', jurado[cid]['particion'], jurado[cid]['voto'], jurado[cid]['jueces_de_acuerdo'], 'todos', 'todos'])
    for k in TOPES:                                                  # las tres tandas quedan registradas desde hoy
        for cid in tandas[k]:
            w.writerow([cid, 'testgold', k, jurado[cid]['particion'], jurado[cid]['voto'], jurado[cid]['jueces_de_acuerdo'], *pareja_de[cid]])
print('Archivos en', SALIDA)
for r in sorted(SALIDA.glob('*.xlsx')):
    print(' ', r.name)

## Qué sigue

1. **Compartir solo los Excel** de cada persona (la carpeta del grupo en OneDrive sirve). `clave_coordinador_v2.csv` y `traducciones.jsonl` se quedan con el coordinador.
2. **Calibración (opcional):** el equipo ya hizo una calibración en octubre, así que `Calibracion3_<Apellido>.xlsx` queda opcional. Las dos reglas que se iban a acordar (Ambiguo y la diferencia entre Basis y Evidence) están en la hoja `Guia_rapida`.
3. **Tanda 1:** cada uno llena su `TestGold_T1_v2_<Apellido>.xlsx` (unas 208 citas). Cuando cada persona lleva la mitad se revisa el acuerdo con las citas que ya tienen dos anotaciones; si sale bajo, se hace una llamada corta solo con los desacuerdos. Al final se calcula el acuerdo (κ por pareja y α de Krippendorff) y un tercero resuelve los desacuerdos. En las hojas T1 v2 la columna Estado no exige justificación en las filas Evaluable; igual se escribe una frase en cada fila, como pide la guía rápida. Desde la tanda 2 la fórmula ya la exige.
4. **Tandas 2 y 3:** ya quedaron escogidas y traducidas. Para generar sus hojas se cambia `TANDA_ACTUAL` a 2 (o 3) y se ejecutan de nuevo los Pasos 1, 2, 3 y 5; el Paso 4 no vuelve a traducir lo que ya está.
4. **En el artículo** se declara que los anotadores contaron con una traducción automática como apoyo y que decidieron con el texto original.